# Ask Coresight v4: a small reranker taught by a large one (about 60-80 minutes on an A100, ~8-12 compute units)

**Why:** live on staging, Ask's instant answer was right 52% of the time and called the AI for 80% of questions. The large
labs answer fast and sure with **two stages**: a fast search finds candidates, a small **reranker** reads the question with
each candidate and decides. Our test showed a large reranker lifts hand-written accuracy from 58% to 70%; this notebook
teaches a small, fast one (22M weights, a few milliseconds on our CPU) to do the same.

1. Runtime → Change runtime type → **A100** → Save. 2. Runtime → **Run all**. At **Choose Files**, pick `catalogue.json` and
`synthetic.json` (both in `Downloads/cip-colab`; hold ⌘ to pick both). 3. **Keep the laptop open.** `synthetic-v4.json` downloads
after step 1 (keep it: a re-run uploads it and skips the writing), `ask-v4.zip` at the end; the GPU is released 15 minutes later.

In [ ]:
# quiet: Colab's own packages are not used here, so pip's notes about them are hidden
!pip -q install "sentence-transformers==3.3.1" accelerate onnx onnxscript onnxruntime > /dev/null 2>&1   # 3.3.1: trained here on 12 Nov
import torch, json, random, time, os, re, numpy as np
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none: Runtime → Change runtime type → GPU")
DEVICE = "cuda"

In [ ]:
from google.colab import files
if not (os.path.exists("catalogue.json") and os.path.exists("synthetic.json")):
    files.upload()   # catalogue.json and synthetic.json (and synthetic-v4.json if an earlier run downloaded it)
data = json.load(open("catalogue.json"))
docs, golden = data["docs"], data["golden"]
TRAIN_ASKS = 6   # asks 1-6 train; asks 7-8 and the golden questions are the test, never trained on
v3 = json.load(open("synthetic.json"))
for d in docs:
    d["v3"] = v3.get(d["key"], [])
print(len(docs), "survey questions ·", len(golden), "golden ·", sum(len(d["v3"]) for d in docs), "practice questions from v3")

In [ ]:
from sentence_transformers import SentenceTransformer

def index_texts():
    owners, texts = [], []
    for n, d in enumerate(docs):
        for t in [d["embed"]] + d["asks"][:TRAIN_ASKS]:
            owners.append(n); texts.append(t)
    return np.array(owners), texts

def ranked(encode, queries, k=10):
    """For each query: its k nearest survey questions (by their best text) and their closeness, in chunks."""
    owners, texts = index_texts()
    vec = encode(texts); out = []
    for s in range(0, len(queries), 2048):
        for row in encode(queries[s:s + 2048]) @ vec.T:
            per = np.full(len(docs), -1.0); np.maximum.at(per, owners, row)
            top = np.argsort(-per)[:k]; out.append((top, per[top]))
    return out

def st_encoder(model):
    return lambda texts: model.encode(texts, batch_size=256, normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=False)

TESTS = {"unseen": ([a for d in docs for a in d["asks"][TRAIN_ASKS:]], [n for n, d in enumerate(docs) for _ in d["asks"][TRAIN_ASKS:]]),
         "golden": ([g["q"] for g in golden], golden)}
def right(name, top, want):
    return docs[top]["group"] == docs[want]["group"] if name == "unseen" else any(w in docs[top]["text"].lower() for w in want["want"])

base = SentenceTransformer("BAAI/bge-small-en-v1.5", device=DEVICE)
NEAREST = {name: ranked(st_encoder(base), queries) for name, (queries, _) in TESTS.items()}
for name, (queries, wants) in TESTS.items():
    print(f"today's search, {name}: right {np.mean([right(name, top[0], w) for (top, _), w in zip(NEAREST[name], wants)]):.1%}")

In [ ]:
# 1. Qwen2.5-7B-Instruct (Apache 2.0) writes practice questions of every kind, and contrast questions that fit this survey
# question but not its closest different look-alike
if not os.path.exists("synthetic-v4.json"):
    from transformers import AutoModelForCausalLM, AutoTokenizer
    name = "Qwen/Qwen2.5-7B-Instruct"
    tok = AutoTokenizer.from_pretrained(name, padding_side="left")
    llm = AutoModelForCausalLM.from_pretrained(name, torch_dtype=torch.bfloat16, device_map="cuda")
    KINDS = ("You write what people type into a consumer-survey search box. For the survey question below, write 16 short questions "
             "someone would type to get THIS question's figures, one of each kind, in this order: which/what are the top ones; "
             "what share or how many people; yes-or-no about people (do/are/have people…); why or the reasons (only if it asks why, "
             "else another share question); how much or what amount (only if it asks an amount, else another which question); "
             "over time (trend, since a year, by month or quarter); by generation; by income or region; men versus women; "
             "comparing two years or seasons; about one of its answer options by name; a retail client's decision; an analyst's "
             "precise wording; a colleague's casual wording; a casual wording with a typo; a very short one (3-5 words). Keep what "
             "kind of answer the survey question gives (a list, yes/no, a reason, an amount): never ask something another kind of "
             "question would answer. Never copy the survey wording, no numbers from the answers. Reply with a JSON list of 16 strings only.")
    CONTRAST = ("Two consumer-survey questions look alike but ask different things. Write 6 short questions someone would type that "
                "QUESTION A answers and QUESTION B does not (use what makes A different: a list vs yes/no, a reason, an amount, the "
                "event, the period, the retailer). Plain words, never copy the survey wording. Reply with a JSON list of 6 strings only.")
    near = ranked(st_encoder(base), [d["embed"] for d in docs], k=8)
    def sibling(n):
        top = near[n][0]
        return next((int(o) for o in top if docs[o]["group"] != docs[n]["group"]), None)
    def chat(system, user):
        return tok.apply_chat_template([{"role": "system", "content": system}, {"role": "user", "content": user}], tokenize=False, add_generation_prompt=True)
    def options(d):
        return "; ".join(a[:40] for a in d["embed"].split("| answers: ")[-1].split("; ")[:8]) if "| answers:" in d["embed"] else ""
    jobs = [(d["key"], "kinds", chat(KINDS, f"Survey question: {d['text'][:300]}\nAnswer options: {options(d)}")) for d in docs]
    for n, d in enumerate(docs):
        s = sibling(n)
        if s is not None:
            jobs.append((d["key"], "contrast", chat(CONTRAST, f"QUESTION A: {d['text'][:250]}\nA's answers: {options(d)}\n"
                                                              f"QUESTION B: {docs[s]['text'][:250]}\nB's answers: {options(docs[s])}")))
    jobs.sort(key=lambda j: len(j[2]))
    synthetic, started = {}, time.time()
    for s in range(0, len(jobs), 96):
        part = jobs[s:s + 96]
        enc = tok([j[2] for j in part], return_tensors="pt", padding=True).to("cuda")
        with torch.no_grad():
            out = llm.generate(**enc, max_new_tokens=480, do_sample=True, temperature=0.7, top_p=0.9, pad_token_id=tok.eos_token_id)
        for (key, kind, _), seq in zip(part, out[:, enc["input_ids"].shape[1]:]):
            text = tok.decode(seq, skip_special_tokens=True)
            try:
                asks = [a.strip() for a in json.loads(text[text.index("["):text.rindex("]") + 1]) if isinstance(a, str) and 6 < len(a.strip()) < 160]
            except ValueError:
                asks = [m.strip() for m in re.findall(r'"([^"]{6,160})"', text)]
            synthetic.setdefault(key, []).extend(asks[:16])
        if s // 96 % 5 == 0:
            print(f"{min(s + 96, len(jobs))} of {len(jobs)} · {time.time() - started:.0f}s", end=" | ")
    del llm; torch.cuda.empty_cache()
    json.dump(synthetic, open("synthetic-v4.json", "w"))
    files.download("synthetic-v4.json")   # keep it: a re-run uploads it and skips this step
v4 = json.load(open("synthetic-v4.json"))
for d in docs:
    d["v4"] = v4.get(d["key"], [])
print("\npractice questions v4:", sum(len(d["v4"]) for d in docs))
for d in random.Random(5).sample(docs, 2):
    print("·", d["text"][:90], "→", d["v4"][:4], d["v4"][-3:])

In [ ]:
# 2. The teacher (bge-reranker-v2-m3) reads a question with each candidate. Is it better than today's search here?
from sentence_transformers import CrossEncoder
teacher = CrossEncoder("BAAI/bge-reranker-v2-m3", device=DEVICE, max_length=256, default_activation_function=torch.nn.Identity(),
                       automodel_args={"torch_dtype": torch.float16})
def teacher_scores(pairs):
    return teacher.predict(pairs, batch_size=512, show_progress_bar=False, convert_to_numpy=True)
results = {}
for name, (queries, wants) in TESTS.items():
    take = range(min(len(queries), 2000 if name == "unseen" else len(queries)))
    t = [teacher_scores([(queries[i], docs[c]["embed"]) for c in NEAREST[name][i][0]]) for i in take]
    results[f"teacher {name}"] = float(np.mean([right(name, NEAREST[name][i][0][int(np.argmax(t[j]))], wants[i]) for j, i in enumerate(take)]))
    print(f"{name}: today's search {np.mean([right(name, NEAREST[name][i][0][0], wants[i]) for i in take]):.1%} · teacher reordering its top 10 {results[f'teacher {name}']:.1%}")

In [ ]:
# 3. Every practice question with its 10 nearest survey questions (and its own), each scored by the teacher. Near-twins
# (the same question in other studies: one group) all count as right
queries = [(t, n) for n, d in enumerate(docs) for t in d["asks"][:TRAIN_ASKS] + [d["name"]] + d["aliases"] + d["v3"] + d["v4"] if t]
random.Random(1).shuffle(queries)
queries = queries[:100000]   # ~1 million teacher readings: ~20 minutes on an A100
near = ranked(st_encoder(base), [q for q, _ in queries], k=10)
lists, started = [], time.time()
for s in range(0, len(queries), 4096):
    chunk = []
    for (q, n), (top, _) in zip(queries[s:s + 4096], near[s:s + 4096]):
        cands = [int(c) for c in top[:10]]
        if n not in cands:   # the right one always among the 10, so the student learns to put it first
            cands[-1] = n
        chunk.append((q, n, cands))
    logits = teacher_scores([(q, docs[c]["embed"]) for q, _, cands in chunk for c in cands])
    at = 0
    for q, n, cands in chunk:
        lists.append((q, cands, logits[at:at + len(cands)].astype(np.float32), [docs[c]["group"] == docs[n]["group"] for c in cands]))
        at += len(cands)
    print(f"{min(s + 4096, len(queries))} of {len(queries)} · {time.time() - started:.0f}s", end=" | ")
del teacher; torch.cuda.empty_cache()
print("\n", len(lists), "practice questions with teacher-scored candidates")

In [ ]:
# 4. The student: a small cross-encoder (MiniLM-L6, 22M weights, Apache 2.0) taught to order candidates as the teacher does,
# mixed with what is right (listwise: half the teacher's spread over the 10, half on the right ones)
from transformers import AutoModelForSequenceClassification, AutoTokenizer
STUDENT = "cross-encoder/ms-marco-MiniLM-L-6-v2"
stok = AutoTokenizer.from_pretrained(STUDENT)
student = AutoModelForSequenceClassification.from_pretrained(STUDENT, num_labels=1).to(DEVICE)
def student_logits(model, pairs):
    enc = stok([p[0] for p in pairs], [p[1] for p in pairs], truncation=True, max_length=256, padding=True, return_tensors="pt").to(DEVICE)
    return model(**enc).logits.view(-1)
EPOCHS, QUERIES, TEMP = 2, 16, 2.0
opt = torch.optim.AdamW(student.parameters(), lr=3e-5)
steps = EPOCHS * (len(lists) // QUERIES)
sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0, (s + 1) / (steps // 10 + 1)) * max(0.0, 1 - s / steps))
student.train(); started = time.time(); step = 0
for epoch in range(EPOCHS):
    random.Random(epoch).shuffle(lists)
    for s in range(0, len(lists) - QUERIES + 1, QUERIES):
        part = [l for l in lists[s:s + QUERIES] if len(l[1]) == 10]
        if not part: continue
        with torch.autocast("cuda", dtype=torch.bfloat16):
            out = student_logits(student, [(q, docs[c]["embed"]) for q, cands, _, _ in part for c in cands]).float().view(len(part), 10)
        teach = torch.softmax(torch.tensor(np.stack([l[2] for l in part]), device=DEVICE) / TEMP, dim=-1)
        hard = torch.tensor(np.stack([l[3] for l in part]), device=DEVICE, dtype=torch.float32)
        target = 0.5 * teach + 0.5 * hard / hard.sum(-1, keepdim=True).clamp(min=1)
        loss = torch.nn.functional.kl_div(torch.log_softmax(out, -1), target, reduction="batchmean")
        loss.backward(); opt.step(); sched.step(); opt.zero_grad(); step += 1
        if step % 2000 == 0: print(f"step {step}/{steps} loss {loss.item():.3f} · {time.time() - started:.0f}s", end=" | ")
student.eval()
print(f"\ntrained in {time.time() - started:.0f}s")

In [ ]:
# 5. How good is it: today's search, then the student reordering the nearest 5 (as Ask will), and how sure it is
def student_probs(pairs):
    with torch.no_grad():
        return torch.sigmoid(student_logits(student, pairs).float()).cpu().numpy()
for name, (queries, wants) in TESTS.items():
    rows = []
    for i in range(len(queries)):
        top = NEAREST[name][i][0][:5]
        p = student_probs([(queries[i], docs[c]["embed"]) for c in top])
        order = np.argsort(-p); best = top[order[0]]
        other = next((p[j] for j in order[1:] if docs[top[j]]["group"] != docs[best]["group"]), 0.0)
        rows.append((p[order[0]] - other, right(name, best, wants[i])))
    rows = np.array(rows, float)
    results[f"student {name}"] = {g: (float(rows[rows[:, 0] >= g, 1].mean()) if (rows[:, 0] >= g).any() else 0.0, float((rows[:, 0] >= g).mean()))
                                  for g in (0.0, 0.05, 0.1, 0.15, 0.2, 0.3)}
    print(f"student {name}: right {rows[:, 1].mean():.1%} · " + " · ".join(f"lead≥{g}: {r:.1%} on {c:.0%}" for g, (r, c) in list(results[f'student {name}'].items())[1:]))

In [ ]:
# 6. Shrink both to the server's format (8-bit ONNX) and check them with the server's own code path (api/ask_model.py)
import onnxruntime, tokenizers, shutil
from onnxruntime.quantization import quantize_dynamic, QuantType
os.makedirs("ask-v4/ask-reranker", exist_ok=True); os.makedirs("ask-v4/bge-small-int8", exist_ok=True)
names = ["input_ids", "attention_mask", "token_type_ids"]
def export(model, dummy, out_names, path):
    torch.onnx.export(model.cpu().eval(), tuple(dummy[k] for k in names), "full.onnx", input_names=names, output_names=out_names,
                      dynamic_axes={k: {0: "batch", 1: "tokens"} for k in names} | {out_names[0]: {0: "batch"}}, opset_version=17, dynamo=False)
    quantize_dynamic("full.onnx", path, weight_type=QuantType.QInt8)
options = onnxruntime.SessionOptions(); options.intra_op_num_threads = options.inter_op_num_threads = 1
# the reranker
export(student, stok(["a question"], ["a survey question"], return_tensors="pt"), ["logits"], "ask-v4/ask-reranker/model_optimized.onnx")
stok.save_pretrained("tok_r"); shutil.copy("tok_r/tokenizer.json", "ask-v4/ask-reranker/tokenizer.json")
rw = tokenizers.Tokenizer.from_file("ask-v4/ask-reranker/tokenizer.json"); rw.enable_truncation(256); rw.enable_padding(pad_id=0, pad_token="[PAD]")
rs = onnxruntime.InferenceSession("ask-v4/ask-reranker/model_optimized.onnx", options, providers=["CPUExecutionProvider"])
def server_rerank(question, texts):   # api/ask_model.py rerank, line for line
    coded = rw.encode_batch([(question, t) for t in texts]); ids = np.array([c.ids for c in coded], np.int64)
    feed = {"input_ids": ids, "attention_mask": np.array([c.attention_mask for c in coded], np.int64), "token_type_ids": np.array([c.type_ids for c in coded], np.int64)}
    return 1 / (1 + np.exp(-rs.run(None, feed)[0].reshape(-1)))
queries, wants = TESTS["golden"]; ok = 0; started = time.time()
for i, q in enumerate(queries):
    top = NEAREST["golden"][i][0][:5]
    ok += right("golden", top[int(np.argmax(server_rerank(q, [docs[c]["embed"] for c in top])))], wants[i])
results["reranker 8-bit golden"] = ok / len(queries)
print(f"reranker 8-bit, server code: golden right {ok / len(queries):.1%} · {(time.time() - started) / len(queries) * 1000:.1f} ms for 5 candidates on one CPU thread")
# today's search model, 8-bit
bert = base[0].auto_model
export(bert, base.tokenize(["a question"]), ["last_hidden_state"], "ask-v4/bge-small-int8/model_optimized.onnx")
base.tokenizer.save_pretrained("tok_b"); shutil.copy("tok_b/tokenizer.json", "ask-v4/bge-small-int8/tokenizer.json")
bw = tokenizers.Tokenizer.from_file("ask-v4/bge-small-int8/tokenizer.json"); bw.enable_truncation(512); bw.enable_padding(pad_id=0, pad_token="[PAD]")
bs = onnxruntime.InferenceSession("ask-v4/bge-small-int8/model_optimized.onnx", options, providers=["CPUExecutionProvider"])
def server_embed(texts):   # api/ask_model.py embed
    out = np.zeros((len(texts), 384), np.float32)
    for s in range(0, len(texts), 32):
        coded = bw.encode_batch(texts[s:s + 32]); ids = np.array([c.ids for c in coded], np.int64)
        out[s:s + 32] = bs.run(None, {"input_ids": ids, "attention_mask": np.array([c.attention_mask for c in coded], np.int64), "token_type_ids": np.zeros_like(ids)})[0][:, 0]
    return out / np.maximum(np.linalg.norm(out, axis=1, keepdims=True), 1e-9)
base.to(DEVICE)
for name, (queries, wants) in TESTS.items():
    results[f"search 8-bit {name}"] = float(np.mean([right(name, top[0], w) for (top, _), w in zip(ranked(server_embed, queries, k=1), wants)]))
    print(f"today's search 8-bit, {name}: right {results[f'search 8-bit {name}']:.1%} (full size: see above)")
print("file sizes:", {f: f"{os.path.getsize(f) / 1e6:.0f} MB" for f in ["ask-v4/ask-reranker/model_optimized.onnx", "ask-v4/bge-small-int8/model_optimized.onnx"]})

In [ ]:
json.dump(results, open("ask-v4/results.json", "w"), indent=1, default=str)
shutil.copy("synthetic-v4.json", "ask-v4/")
shutil.make_archive("ask-v4", "zip", "ask-v4")
print("READY: ask-v4.zip is downloading to your Downloads folder.")
print("If the download does not start: left bar → folder icon → content → ask-v4.zip → ⋮ → Download (within 15 minutes).")
files.download("ask-v4.zip")

In [ ]:
# Stop paying once the download has had plenty of time: an idle GPU still uses units
import time
from google.colab import runtime
print("Releasing the GPU in 15 minutes. If ask-v4.zip is not in Downloads yet: folder icon → content → ask-v4.zip → ⋮ → Download")
time.sleep(900)
runtime.unassign()